[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ofermend/hands-on-rag/blob/main/chapter2/pgvector-simple.ipynb)

# Chapter 2: Vector Search with pgvector

This notebook demonstrates how to use [pgvector](https://github.com/pgvector/pgvector), the open-source vector similarity search extension for PostgreSQL, to store sentence embeddings and perform nearest-neighbor queries. We embed sentences using Sentence Transformers, insert them into a pgvector-enabled table, and run cosine similarity searches directly in SQL.

**What you'll learn:**
- How to create a pgvector table and HNSW index in PostgreSQL
- How to encode text into embeddings with Sentence Transformers and store them
- How to perform vector similarity search using SQL

**Prerequisites:**
- A running PostgreSQL instance with the `pgvector` extension installed
- `pip install -r requirements.txt`
- Set `PGVECTOR_PASSWORD` environment variable (e.g. `export PGVECTOR_PASSWORD=yourpassword`)

> **Note:** This notebook requires a local PostgreSQL server and will not run out-of-the-box on Google Colab.

### Using PG-Vector for vector search

In this notebook, we will show how to build a vector database in PGVector and commerce a vector search. 

First, we embed a few example sentences using Sentence Transformers. We use the `all-MiniLM-L6-v2` model to turn short sentences like "I am a happy person." into dense numerical vectors. These embeddings will go into the vector database to be looked up later. 

In [1]:
from typing import List
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

model = SentenceTransformer('all-MiniLM-L6-v2')

sentences = [
    "I am a happy person.",
    "I am a joyful person.",
    "I am a pessimistic person.",
    "I am not an optimistic person."
]
embeddings = model.encode(sentences)

The resulting `embeddings` object is a 2D NumPy array where each row corresponds to a sentence and each column corresponds to one embedding dimension (384 dimensions for `all-MiniLM-L6-v2`). Later, we will store these embeddings in PostgreSQL using the `pgvector` extension so that we can run similarity search directly in the database.

Then, connect to a PostgreSQL instance and create a database called `ragbook`.
We operate on the default database called `postgres` which is the default database in any PG instance. It is the place for performaing administrative tasks. After creating the `ragbook` database, we will switch to `ragbook` for our vector ingestion and serach. 

In [2]:
import psycopg2
import os

# Database connection parameters
pg_password = os.environ.get('PGVECTOR_PASSWORD', 'RAGBOOK')
DB_CONFIG = {
    'host': 'localhost',
    'port': 5432,
    'user': 'postgres',
    'password': pg_password,
    'database': 'postgres'  # Connect to default database first
}

# Connect to PostgreSQL
conn = psycopg2.connect(**DB_CONFIG)
conn.autocommit = True
cursor = conn.cursor()

DB_FOR_VECTORS = "ragbook" # lowercase only, use a non-default database for our experiments

# Create a database if it doesn't exist
try: 
    cursor.execute(f"CREATE DATABASE {DB_FOR_VECTORS};")
except psycopg2.errors.DuplicateDatabase:
    print (f'Database {DB_FOR_VECTORS} already exists.')



We now switch to the new database.

In [3]:
# Switch to the new database
DB_CONFIG['database'] = DB_FOR_VECTORS
conn = psycopg2.connect(**DB_CONFIG)
conn.autocommit = True
cursor = conn.cursor()  # Create a new cursor connecting to the new db `DB_FOR_VECTORS`

We now create a table called `sentence_embeddings` with two columns: the original sentence text and its embedding. The `embedding` column uses the `VECTOR(384)` type, because the `all-MiniLM-L6-v2` model produces 384-dimensional embeddings.

After creating the table, we enable the `vector` extension (if it is not already enabled) and define an HNSW index on the `embedding` column using `vector_l2_ops`. This index structure allows PostgreSQL to perform fast approximate nearest-neighbor search over high-dimensional vectors.

In [4]:
# Create a table for storing vectors

# Enable PG-Vector extension
cursor.execute("CREATE EXTENSION IF NOT EXISTS vector;")

# Drop the table if it already exists
cursor.execute("DROP TABLE IF EXISTS sentence_embeddings;")

# Create a table called `sentence_embeddings`
cursor.execute("""
    CREATE TABLE IF NOT EXISTS sentence_embeddings (
        text TEXT NOT NULL,
        embedding VECTOR(384)  -- all-MiniLM-L6-v2 produces 384-dimensional embeddings
    );
""")

# Create HNSW index in the table `sentence_embeddings` for efficient similarity search
cursor.execute("""
    CREATE INDEX 
    ON sentence_embeddings 
    USING hnsw (embedding vector_l2_ops) 
    WITH (m = 16, ef_construction = 64);
""")


> **Note:** HNSW (Hierarchical Navigable Small World) index parameters control the speed/accuracy trade-off. **m** sets the number of bi-directional links per node — higher values improve recall but use more memory. **ef_construction** controls how many candidates are considered during index building — higher values produce a better-quality graph at the cost of longer build times. The defaults here (m=16, ef_construction=64) work well for small-to-medium datasets.

With the table `sentence_embeddings` ready, we can insert the precomputed embeddings into it. We will do it one embedding each time using the SQLs `INSERT` statement.  Note that the variable `embeddings` we obtained earlier are 2D NumPy ndarrays. When inserting into a PG table, each row needs to be converted into a list of floats and then serialized into a string for properly forming an `INSERT` statement as a string. 

In [5]:
# Insert sample sentences and their embeddings into the table `sentence_embeddings`

for sentence, embedding in zip(sentences, embeddings):
        embedding_as_list: List[float] = embedding.tolist()
        cursor.execute(
            "INSERT INTO sentence_embeddings (text, embedding) VALUES (%s, %s::vector)",
            (sentence, embedding_as_list)
        )

Now we can take a sneak peek at the table contents. Each row is a sentence and its embedding. 

In [6]:
# Take a sneak peek at the table contents
cursor.execute("SELECT text, embedding FROM sentence_embeddings;")
rows = cursor.fetchall()
for row in rows:
    print(row)

('I am a happy person.', '[0.004647262,0.06651061,0.01479143,-0.029557081,-0.03556158,-0.027606945,0.12540376,0.042491496,0.012504493,-0.0063209925,0.084500745,-0.009377444,0.011229956,-0.028832374,0.05915539,0.017892506,-0.041411843,-0.10653302,0.008443907,0.019869447,-0.084468566,0.0074553145,-0.015888017,0.026268711,-0.026523223,0.020369602,0.049003474,-0.017068246,0.030692995,0.050624087,-0.051756352,-0.009287611,0.11134335,0.0022187831,-0.025793161,0.0023096574,-0.05748574,-0.06991997,0.011813442,0.050560832,0.015797134,-0.011647742,-0.02562016,-0.0073235696,0.016757397,-0.025864089,0.07737778,-0.013637424,0.061339527,-0.063298434,-0.0039380025,0.09201368,-0.07445012,-0.009040345,0.051146287,0.018042333,0.06534488,-0.002853362,0.007229924,0.04942291,0.044622984,0.09821895,-0.017557092,0.04602287,0.091186576,-0.0064609814,-0.036592416,-0.021893715,-0.052886486,0.0006522949,-0.020817237,0.006281041,0.09244518,0.014187928,-0.011610401,-0.03512725,0.082290456,0.0049449117,0.06444221,0

Finally, let's do the vector search. To allow searching again and again, we create a function: 

In [7]:
# Embed a query and find the most similar sentence in the database

def vector_search(query, model, top_k):
    query_embedding = model.encode([query])[0]
    query_embedding_as_list: List[float] = query_embedding.tolist()

    cursor.execute(
            """
            SELECT text, 
                    1 - (embedding <=> %s::vector) as similarity
            FROM sentence_embeddings
            WHERE embedding IS NOT NULL
            ORDER BY embedding <=> %s::vector
            LIMIT %s;
            """,
            (query_embedding_as_list, query_embedding_as_list, top_k)
        )

    results = cursor.fetchall()

    for row in results:
        print(row)

query = "I am a smiling person."
vector_search(query, model, 3)

('I am a happy person.', 0.7639916170833)
('I am a joyful person.', 0.693439245223999)
('I am not an optimistic person.', 0.38356050848960876)


The special notation ``vector`` tells PG to treat the field `embedding` as vectors. 
The operation `<=>` performs cosine similarity search. In the SQL `SELECT` statement above, `embedding` refers to the column `embedding` in the table `sentence_embeddings`.

You may wonder why `1 - (embedding <=> %s::vector) as similarity`. This is because in PG Vector, the operation `<=>` in PG-vectgor actually measures the dissimilarity between two vectors. Hence we have to use its complement to measure similarity. 

In the example above, the query sentence is "I am a smiling person." The most similar sentence in the database is "I am a happy person." with a similarity score of 0.76. 

Let's try again. 

In [8]:
query = "I have a bad feeling about the future."
vector_search(query, model, 3)

('I am a pessimistic person.', 0.5613031721891873)
('I am not an optimistic person.', 0.505988389610553)
('I am a happy person.', 0.3224701871696283)


This time, the query is "I have a bad feeling about the future." The most similar sentence in the database is "I am not an optimistic person." with a similarity score of 0.38.  It totally makes sense. 